# Поиск частых наборов: алгоритм Apriori

Программа находит частые наборы объектов (itemsets) в транзакционных данных. Алгоритм реализован самостоятельно, без готовых библиотек ассоциативных правил (`mlxtend`, `apyori` и т.п.).

**Параметры:**
- набор транзакций;
- порог поддержки (`min_support`);
- способ упорядочивания результата: по убыванию поддержки или лексикографически.

Для каждого частого набора выводятся сами объекты и значение поддержки.


## Идея алгоритма

Поддержка набора $X$ — доля транзакций, в которых встречаются все объекты из $X$:

$$\mathrm{supp}(X) = \frac{|\{t \in T : X \subseteq t\}|}{|T|}$$

Набор частый, если $\mathrm{supp}(X) \ge \mathrm{min\_support}$.

Apriori использует свойство антимонотонности: любой поднабор частого набора тоже частый. Поэтому:
1. считаются частые 1-элементные наборы $L_1$;
2. кандидаты $C_k$ строятся соединением наборов из $L_{k-1}$;
3. кандидат отбрасывается, если хотя бы один его $(k-1)$-поднабор не входит в $L_{k-1}$;
4. по данным считается поддержка оставшихся кандидатов, формируется $L_k$;
5. шаги 2–4 повторяются, пока появляются новые частые наборы.


In [ ]:
from collections import defaultdict
from itertools import combinations
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11
plt.rcParams["font.family"] = "DejaVu Sans"


## Загрузка данных

Файл `baskets.csv`: одна транзакция (покупка) на строку, товары через запятую. Кодировка Windows-1251.


In [ ]:
def load_transactions(path: str | Path, encoding: str = "cp1251") -> list[set[str]]:
    """Читает транзакции: каждая непустая строка — набор товаров."""
    transactions: list[set[str]] = []
    with open(path, encoding=encoding) as f:
        for line in f:
            items = {item.strip() for item in line.strip().split(",") if item.strip()}
            if items:
                transactions.append(items)
    return transactions


DATA_PATH = Path("baskets.csv")
transactions = load_transactions(DATA_PATH)

all_items = {item for t in transactions for item in t}
print(f"Транзакций: {len(transactions)}")
print(f"Уникальных товаров: {len(all_items)}")
print(f"Средняя длина корзины: {sum(len(t) for t in transactions) / len(transactions):.2f}")
print("Пример первой транзакции:", sorted(transactions[0])[:8], "...")


## Реализация Apriori


In [ ]:
def _apriori_join(prev_level: list[frozenset[str]], k: int) -> list[frozenset[str]]:
    """C_k: объединение пар из L_{k-1}, дающих набор длины k."""
    candidates: set[frozenset[str]] = set()
    m = len(prev_level)
    for i in range(m):
        for j in range(i + 1, m):
            union = prev_level[i] | prev_level[j]
            if len(union) == k:
                candidates.add(union)
    return list(candidates)


def _apriori_prune(
    candidates: list[frozenset[str]],
    prev_frequent: set[frozenset[str]],
    k: int,
) -> list[frozenset[str]]:
    """Отбросить кандидата, у которого есть нечастый (k-1)-поднабор."""
    kept = []
    for cand in candidates:
        if all(frozenset(sub) in prev_frequent for sub in combinations(cand, k - 1)):
            kept.append(cand)
    return kept


def apriori(
    transactions: list[set[str]],
    min_support: float,
    order: str = "support_desc",
) -> list[tuple[frozenset[str], float]]:
    """Поиск частых наборов алгоритмом Apriori.

    Параметры
    ---------
    transactions : список транзакций (каждая — множество строк-товаров).
    min_support  : порог поддержки в долях, например 0.05 = 5%.
    order        : "support_desc" — по убыванию поддержки (при равенстве лексикографически);
                   "lex" — лексикографически по составу набора.

    Возвращает
    ----------
    Список пар (набор как frozenset, поддержка), отсортированный заданным способом.
    """
    if order not in {"support_desc", "lex"}:
        raise ValueError('order должен быть "support_desc" или "lex"')

    n_trans = len(transactions)
    if n_trans == 0:
        return []

    min_count = min_support * n_trans
    support: dict[frozenset[str], float] = {}

    item_counts: dict[str, int] = defaultdict(int)
    for basket in transactions:
        for item in basket:
            item_counts[item] += 1

    L_current: list[frozenset[str]] = []
    for item, cnt in item_counts.items():
        if cnt >= min_count:
            itemset = frozenset([item])
            L_current.append(itemset)
            support[itemset] = cnt / n_trans
    L_current.sort(key=lambda s: tuple(sorted(s)))

    all_frequent = list(L_current)
    k = 2
    while L_current:
        candidates = _apriori_join(L_current, k)
        candidates = _apriori_prune(candidates, set(L_current), k)
        if not candidates:
            break

        cand_index = set(candidates)
        counts: dict[frozenset[str], int] = defaultdict(int)
        for basket in transactions:
            if len(basket) < k:
                continue
            for comb in combinations(sorted(basket), k):
                itemset = frozenset(comb)
                if itemset in cand_index:
                    counts[itemset] += 1

        L_next: list[frozenset[str]] = []
        for cand in candidates:
            cnt = counts[cand]
            if cnt >= min_count:
                L_next.append(cand)
                support[cand] = cnt / n_trans

        L_next.sort(key=lambda s: tuple(sorted(s)))
        all_frequent.extend(L_next)
        L_current = L_next
        k += 1

    result = [(itemset, support[itemset]) for itemset in all_frequent]
    if order == "support_desc":
        result.sort(key=lambda x: (-x[1], tuple(sorted(x[0]))))
    else:
        result.sort(key=lambda x: tuple(sorted(x[0])))
    return result


def itemsets_to_frame(itemsets: list[tuple[frozenset[str], float]]) -> pd.DataFrame:
    """Таблица: набор, длина, поддержка."""
    rows = []
    for itemset, supp in itemsets:
        items = ", ".join(sorted(itemset))
        rows.append({"набор": items, "длина": len(itemset), "поддержка": supp})
    return pd.DataFrame(rows)


## Пример работы программы

Порог поддержки 5%, два способа сортировки. Для наглядности показываются первые строки таблицы.


In [ ]:
MIN_SUPPORT_DEMO = 0.05

frequent_by_support = apriori(transactions, MIN_SUPPORT_DEMO, order="support_desc")
frequent_lex = apriori(transactions, MIN_SUPPORT_DEMO, order="lex")

print(f"Число частых наборов при min_support={MIN_SUPPORT_DEMO:.0%}: {len(frequent_by_support)}\n")

print("Сортировка по убыванию поддержки:")
display(
    itemsets_to_frame(frequent_by_support)
    .style.format({"поддержка": "{:.2%}"})
)

print("\nСортировка лексикографическая (первые 15 строк):")
display(
    itemsets_to_frame(frequent_lex)
    .head(15)
    .style.format({"поддержка": "{:.2%}"})
)


## Эксперименты на `baskets.csv`

Порог поддержки: 1%, 3%, 5%, 10%, 15%. Для каждого значения измеряется время работы и считается, сколько частых наборов каждой длины найдено.


In [ ]:
THRESHOLDS = [0.01, 0.03, 0.05, 0.10, 0.15]

experiment_rows = []
length_rows = []

for thr in THRESHOLDS:
    t0 = perf_counter()
    found = apriori(transactions, thr, order="support_desc")
    elapsed = perf_counter() - t0

    by_len: dict[int, int] = defaultdict(int)
    for itemset, _ in found:
        by_len[len(itemset)] += 1

    experiment_rows.append(
        {
            "порог поддержки": f"{thr:.0%}",
            "min_support": thr,
            "число частых наборов": len(found),
            "макс. длина": max(by_len) if by_len else 0,
            "время, с": elapsed,
        }
    )
    for length, count in sorted(by_len.items()):
        length_rows.append(
            {
                "порог поддержки": f"{thr:.0%}",
                "min_support": thr,
                "длина набора": length,
                "количество": count,
            }
        )

exp_df = pd.DataFrame(experiment_rows)
len_df = pd.DataFrame(length_rows)

print("Сводка экспериментов:")
display(exp_df.drop(columns=["min_support"]).style.format({"время, с": "{:.3f}"}))

print("\nКоличество частых наборов по длине:")
pivot = (
    len_df.pivot_table(
        index="порог поддержки",
        columns="длина набора",
        values="количество",
        fill_value=0,
        aggfunc="sum",
    )
    .reindex([f"{t:.0%}" for t in THRESHOLDS])
    .astype(int)
)
display(pivot)


## Визуализация

### Быстродействие при изменяемом пороге поддержки


In [ ]:
fig, ax = plt.subplots()
ax.plot(
    exp_df["min_support"] * 100,
    exp_df["время, с"],
    marker="o",
    linewidth=2,
)
ax.set_title("Время работы Apriori на baskets.csv")
ax.set_xlabel("Порог поддержки, %")
ax.set_ylabel("Время, с")
ax.set_xticks(exp_df["min_support"] * 100)
for x, y in zip(exp_df["min_support"] * 100, exp_df["время, с"]):
    ax.annotate(f"{y:.3f} с", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
fig.tight_layout()
plt.show()


### Число частых наборов разной длины при изменяемом пороге поддержки


In [ ]:
pivot_plot = (
    len_df.pivot_table(
        index="min_support",
        columns="длина набора",
        values="количество",
        fill_value=0,
        aggfunc="sum",
    )
    .reindex(THRESHOLDS)
)

fig, ax = plt.subplots()
x = range(len(THRESHOLDS))
width = 0.8 / max(len(pivot_plot.columns), 1)

for i, length in enumerate(pivot_plot.columns):
    values = pivot_plot[length].to_numpy()
    positions = [p + (i - (len(pivot_plot.columns) - 1) / 2) * width for p in x]
    ax.bar(positions, values, width=width, label=f"длина {int(length)}")
    for pos, val in zip(positions, values):
        if val > 0:
            ax.text(pos, val, str(int(val)), ha="center", va="bottom", fontsize=9)

ax.set_title("Число частых наборов по длине (baskets.csv)")
ax.set_xlabel("Порог поддержки, %")
ax.set_ylabel("Количество частых наборов")
ax.set_xticks(list(x))
ax.set_xticklabels([f"{t:.0%}" for t in THRESHOLDS])
ax.legend()
fig.tight_layout()
plt.show()


## Краткий вывод

- При **низком** пороге поддержки (1%) частых наборов больше, среди них появляются пары и тройки, а перебор кандидатов занимает больше времени.
- При **повышении** порога (10–15%) остаются в основном одиночные популярные товары, число наборов резко падает, алгоритм отрабатывает быстрее: меньше $L_1$, меньше соединений и проходов по данным.
- Это ожидаемое поведение Apriori: стоимость шага определяется числом кандидатов, а оно растёт при ослаблении порога.
